# Level 1: Can models hand off at all?

This notebook answers one yes/no question per pair of models: **can model A start an image mid-generation and model B finish it, without the result breaking?**

What it does, in order:

1. Prints the **format table** (VAE config, weight fingerprints, scheduler) for SDXL, SSD-1B, Vega, Playground v2.5 and SD 1.5. No GPU needed.
2. Loads SDXL + SSD-1B once (shared text encoders + fp16-fix VAE) and times each alone.
3. **Verifies VAE compatibility empirically**: decodes an SSD-1B latent with three different VAE copies and compares.
4. Does one handoff in each direction at 40% (Method 1 small→large, Method 2 large→small) and builds the timing table.
5. Checks that the one-pipeline UNet swap and the two-pipeline `denoising_end/denoising_start` route give the same image.
6. Same-company pair (SSD-1B ↔ Vega), then the pairs that should **break** (SDXL → Playground v2.5, SD 1.5 → SDXL) and the **pixel bridge** that rescues them.

**Gate:** a clean handoff image in each direction, a timing table, and a list of which pairs work.

Time on a T4: about 25 minutes for steps 1–5 (plus ~15 GB of downloads the first time). Step 6 is optional and downloads another ~15 GB.


In [ ]:
# --- Setup (Colab: Runtime > Change runtime type > T4 GPU or better) ---
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""   # paste your GitHub URL here once the repo is pushed; leave empty to upload the zip instead
if IN_COLAB and not os.path.exists("image-rerouter"):
    if REPO_URL:
        !git clone -q $REPO_URL
    else:
        from google.colab import files
        print("Pick the image-rerouter zip (any name, e.g. image-rerouter_2026-10-26.zip) in the file dialog below.")
        ZIP = next(iter(files.upload()))
        !unzip -q -o "$ZIP" && rm "$ZIP"
if IN_COLAB:
    %cd image-rerouter
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao   # Colab's preinstalled torchao is too old for new diffusers (FqnToConfig import error); unused here
# optional: faster downloads + higher rate limits
# from huggingface_hub import login; login()
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
from rerouter import utils
cfg = utils.load_config()
cfg

## 1. Format table (no GPU)

Analytic check from the Hub. Spoiler, already measured tensor by tensor (see `docs/compat_report.md`):

- **SSD-1B's shipped VAE is `madebyollin/sdxl-vae-fp16-fix`** (max abs difference 0.015 = fp16 rounding). The fp16-fix VAE is SDXL's VAE with rescaled internals, so the **latent space is the same**. The file hash differs from SDXL's, which is exactly the "subtle" case the plan warns about, but here it's benign.
- SSD-1B's and Vega's text encoders differ from SDXL's only by rounding noise (mean |Δ| ≈ 6e-5, a bf16 round-trip). Sharing SDXL's encoders is safe.
- Vega and Playground v2.5 ship SDXL's VAE byte for byte, but Playground uses `scaling_factor` 0.5 (not 0.13025) **and** an EDM schedule.
- SD 1.5 uses a different VAE (and 512px), so its latents are a different language.

In [ ]:
!python bench/compat_check.py
from IPython.display import Markdown, display
display(Markdown(open("docs/compat_report.md").read()))

## 2. Load SDXL + SSD-1B once, time each alone

In [ ]:
from rerouter.models import ModelBank
from rerouter.handoff import generate, warmup
bank = ModelBank(cfg).load(("large", "small"))
print("load seconds:", bank.load_seconds)
print("UNet params (billions):", bank.param_counts())
print(f"GPU memory after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")
warmup(bank, cfg)   # never time the first run

In [ ]:
PROMPT = 'a storefront sign that says "OPEN 24 HOURS", photo'
SEED = 42
STEPS = cfg["generation"]["steps"]
runs = {}
runs["large alone"] = generate(bank, PROMPT, SEED, "large", 0.0, STEPS, cfg)
runs["small alone"] = generate(bank, PROMPT, SEED, "small", 0.0, STEPS, cfg)
for k, r in runs.items():
    print(f"{k:12s} {r.gpu_seconds:6.2f} s  peak {r.peak_mem_gb:.2f} GB  plan {r.plan}")

In [ ]:
import matplotlib.pyplot as plt
def grid(images, titles, cols=None, size=4):
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size*cols, size*rows))
    for ax, im, t in zip(axes.flat if hasattr(axes, "flat") else [axes], images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
grid([r.image for r in runs.values()], list(runs))

## 3. VAE compatibility, empirically

Run SSD-1B to a **latent** (no decode), then decode that one latent with:

- (a) the fp16-fix VAE the bench uses,
- (b) the VAE shipped inside `segmind/SSD-1B`,
- (c) the stock SDXL VAE (in fp32, since it NaNs in fp16).

If all three give the same picture (PSNR > 30 dB), SSD-1B and SDXL speak the same latent language and a mid-generation handoff is meaningful.

In [ ]:
import numpy as np
from diffusers import AutoencoderKL
from diffusers.image_processor import VaeImageProcessor

g = torch.Generator("cuda").manual_seed(SEED)
bank.set_unet("small")
lat = bank.pipe(PROMPT, num_inference_steps=STEPS, guidance_scale=cfg["generation"]["guidance_scale"],
                generator=g, output_type="latent").images
bank.restore()
print("latent shape:", tuple(lat.shape), "(1, 4, 128, 128 expected)")

def decode_with(vae, lat):
    vae = vae.to("cuda")
    with torch.no_grad():
        x = vae.decode((lat / vae.config.scaling_factor).to(vae.dtype)).sample
    img = VaeImageProcessor().postprocess(x.float().cpu(), output_type="pil")[0]
    return img

vae_fix  = bank.pipe.vae
vae_ssd  = AutoencoderKL.from_pretrained(cfg["models"]["small"], subfolder="vae", torch_dtype=torch.float32)
vae_sdxl = AutoencoderKL.from_pretrained(cfg["models"]["large"], subfolder="vae", torch_dtype=torch.float32)
decodes = {"fp16-fix VAE (bench)": decode_with(vae_fix, lat),
           "VAE shipped with SSD-1B": decode_with(vae_ssd, lat),
           "stock SDXL VAE (fp32)": decode_with(vae_sdxl, lat)}
del vae_ssd, vae_sdxl; torch.cuda.empty_cache()

def psnr(a, b):
    a, b = np.asarray(a, dtype=np.float32), np.asarray(b, dtype=np.float32)
    mse = ((a - b) ** 2).mean()
    return 99.0 if mse == 0 else 10 * np.log10(255 ** 2 / mse)
ref = decodes["fp16-fix VAE (bench)"]
for k, im in decodes.items():
    print(f"{k:26s} PSNR vs bench VAE: {psnr(ref, im):5.1f} dB")
grid(list(decodes.values()), list(decodes))
print("Verdict:", "SAME latent space" if all(psnr(ref, im) > 30 for im in decodes.values()) else "DIFFERENT - stop and investigate")

## 4. One handoff in each direction at 40%

`switch_fraction` = fraction of the total steps run by the **first** model.
- Method 1 `s2l@0.4`: SSD-1B runs steps 0–11, SDXL runs 12–29.
- Method 2 `l2s@0.4`: SDXL runs steps 0–11, SSD-1B runs 12–29.

In [ ]:
runs["Method 1: small->large @0.4"] = generate(bank, PROMPT, SEED, "s2l", 0.4, STEPS, cfg)
runs["Method 2: large->small @0.4"] = generate(bank, PROMPT, SEED, "l2s", 0.4, STEPS, cfg)
import pandas as pd
timing = pd.DataFrame([{"run": k, "gpu_seconds": round(r.gpu_seconds, 2), "peak_mem_gb": round(r.peak_mem_gb, 2),
                        "plan": r.plan, "impl": r.impl} for k, r in runs.items()])
timing["saving_vs_large"] = (1 - timing.gpu_seconds / timing.gpu_seconds.iloc[0]).map("{:+.0%}".format)
timing.to_csv("results/level1_timing.csv", index=False)
display(timing)
grid([r.image for r in runs.values()], list(runs), cols=2, size=5)
for k, r in runs.items():
    r.image.save(f"results/level1_{k.split(':')[0].replace(' ', '_').lower()}.png")

## 5. Does the UNet swap equal the two-pipeline route?

Same seed, same handoff point; the only difference is the plumbing. The two images should be nearly identical (small differences come from fp16 and the `denoising_end` timestep rounding). This validates `impl: swap` (which we use because it is cheaper) against the textbook Option A.

In [ ]:
import copy
cfg2 = copy.deepcopy(cfg); cfg2["handoff"]["impl"] = "twopipe"
a = generate(bank, PROMPT, SEED, "s2l", 0.4, STEPS, cfg)
b = generate(bank, PROMPT, SEED, "s2l", 0.4, STEPS, cfg2)
print(f"swap: {a.gpu_seconds:.2f}s   twopipe: {b.gpu_seconds:.2f}s   PSNR between them: {psnr(a.image, b.image):.1f} dB")
grid([a.image, b.image], ["impl=swap", "impl=twopipe"])

## 6. Other pairs

### 6a. Same company: SSD-1B ↔ Vega (should work)
Vega is a smaller distillation that also uses SDXL's VAE. On a 15 GB T4 we free the SDXL UNet first.

In [ ]:
RUN_OPTIONAL = True   # set False to skip the heavy downloads in 6a-6c
if RUN_OPTIONAL:
    import gc
    del bank.unets["large"]; gc.collect(); torch.cuda.empty_cache()
    from diffusers import UNet2DConditionModel
    bank.unets["tiny"] = UNet2DConditionModel.from_pretrained(cfg["models"]["tiny"], subfolder="unet",
                                                               torch_dtype=torch.float16, variant="fp16").to("cuda")
    bank._check_unets_agree()
    from rerouter.schedule import Segment
    pair = {"small alone": generate(bank, PROMPT, SEED, "small", 0.0, STEPS, cfg),
            "tiny alone (Vega)": generate(bank, PROMPT, SEED, "small", 0.0, STEPS, cfg, segments=[Segment("tiny", 0, STEPS)]),
            "small->tiny @0.4": generate(bank, PROMPT, SEED, "s2l", 0.4, STEPS, cfg, segments=[Segment("small", 0, 12), Segment("tiny", 12, STEPS)]),
            "tiny->small @0.4": generate(bank, PROMPT, SEED, "l2s", 0.4, STEPS, cfg, segments=[Segment("tiny", 0, 12), Segment("small", 12, STEPS)])}
    for k, r in pair.items(): print(f"{k:20s} {r.gpu_seconds:6.2f}s")
    grid([r.image for r in pair.values()], list(pair), cols=2, size=5)

### 6b. Different schedule: SDXL → Playground v2.5 (should break; pixel bridge works)

Playground v2.5 uses the same VAE weights but `scaling_factor=0.5` and an **EDM** schedule. Handing it an SDXL latent at "step 12" means nothing to it: it reads the wrong noise level. The naive handoff below should come out blurry/burnt. The **pixel bridge** (decode → img2img at strength 0.6) works for any pair but costs an extra decode+encode, and Playground redraws rather than continues.

In [ ]:
if RUN_OPTIONAL:
    bank.free(); del bank; gc.collect(); torch.cuda.empty_cache()
    from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, AutoencoderKL
    vae = AutoencoderKL.from_pretrained(cfg["models"]["vae"], torch_dtype=torch.float16)
    sdxl = StableDiffusionXLPipeline.from_pretrained(cfg["models"]["large"], vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
    g = torch.Generator("cuda").manual_seed(SEED)
    sdxl_lat = sdxl(PROMPT, num_inference_steps=STEPS, denoising_end=0.4, output_type="latent", generator=g).images
    sdxl_img = sdxl(PROMPT, num_inference_steps=STEPS, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    del sdxl; gc.collect(); torch.cuda.empty_cache()
    pg = StableDiffusionXLImg2ImgPipeline.from_pretrained("playgroundai/playground-v2.5-1024px-aesthetic", torch_dtype=torch.float16, variant="fp16").to("cuda")
    # naive: feed the SDXL latent straight in (rescale 0.13025 -> 0.5 so at least the magnitude is right)
    naive = pg(PROMPT, image=sdxl_lat * (0.5 / 0.13025), denoising_start=0.4, num_inference_steps=STEPS, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    # pixel bridge: decode SDXL's own finished image, let Playground redraw 60% of it
    bridge = pg(PROMPT, image=sdxl_img, strength=0.6, num_inference_steps=STEPS, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    grid([sdxl_img, naive, bridge], ["SDXL alone", "naive latent handoff -> Playground (expect broken)", "pixel bridge -> Playground"])
    del pg; gc.collect(); torch.cuda.empty_cache()

### 6c. Different format: SD 1.5 → SDXL (impossible in latent space; pixel bridge works)

SD 1.5's latent for a 512px image is 64×64×4 in a *different* VAE's language. Decoding it with SDXL's VAE gives garbage. The pixel bridge (decode with SD 1.5's VAE → upscale → SDXL img2img) is the only route.

In [ ]:
if RUN_OPTIONAL:
    from diffusers import StableDiffusionPipeline
    sd15 = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16, safety_checker=None).to("cuda")
    g = torch.Generator("cuda").manual_seed(SEED)
    lat15 = sd15(PROMPT, num_inference_steps=STEPS, output_type="latent", generator=g).images
    img15 = VaeImageProcessor().postprocess(sd15.vae.decode(lat15 / sd15.vae.config.scaling_factor).sample.float().cpu(), output_type="pil")[0]
    del sd15; gc.collect(); torch.cuda.empty_cache()
    # wrong-VAE decode: SDXL's VAE reading SD1.5's latent
    vae = AutoencoderKL.from_pretrained(cfg["models"]["vae"], torch_dtype=torch.float16).to("cuda")
    with torch.no_grad():
        wrong = VaeImageProcessor().postprocess(vae.decode(lat15.half() / 0.13025).sample.float().cpu(), output_type="pil")[0]
    sdxl_i2i = StableDiffusionXLImg2ImgPipeline.from_pretrained(cfg["models"]["large"], vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
    bridge15 = sdxl_i2i(PROMPT, image=img15.resize((1024, 1024)), strength=0.5, num_inference_steps=STEPS, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    grid([img15, wrong, bridge15], ["SD 1.5 alone (512px)", "SD1.5 latent decoded by SDXL VAE (garbage)", "pixel bridge -> SDXL img2img"])
    del sdxl_i2i, vae; gc.collect(); torch.cuda.empty_cache()

## 7. Gate check

Fill in from what you saw above, then save. This is the Level 1 deliverable.

In [ ]:
pairs = pd.DataFrame([
    ("SSD-1B -> SDXL (Method 1)", "latent", "WORKS", "same VAE (fp16-fix), same schedule"),
    ("SDXL -> SSD-1B (Method 2)", "latent", "WORKS", "same VAE (fp16-fix), same schedule"),
    ("SSD-1B <-> Vega",            "latent", "WORKS?", "fill in from 6a"),
    ("SDXL -> Playground v2.5",    "latent", "BREAKS?", "EDM schedule + scaling 0.5; fill in from 6b"),
    ("SDXL -> Playground v2.5",    "pixel bridge", "WORKS?", "extra decode/encode; redraws rather than continues"),
    ("SD 1.5 -> SDXL",             "latent", "IMPOSSIBLE", "different VAE, 64x64 vs 128x128"),
    ("SD 1.5 -> SDXL",             "pixel bridge", "WORKS?", "fill in from 6c"),
], columns=["pair", "route", "result", "why"])
pairs.to_csv("results/level1_pairs.csv", index=False)
display(pairs)
print("Timing table -> results/level1_timing.csv ; images -> results/level1_*.png")